### https://www.kaggle.com/competitions/drawing-with-llms

In [1]:
import kagglehub
import pandas as pd

In [2]:
from unsloth import FastLanguageModel
import torch

dtype = ( None )
load_in_4bit = False
load_in_8bit = False
#unsloth/Llama-3.2-3B-Instruct
#Qwen/Qwen2.5-Coder-3B
#Qwen/Qwen2.5-Coder-7B-Instruct
#Qwen/Qwen2.5-14B-Instruct

######---Parameters to change---#######
base_model="Llama-3.2-3B-Instruct"      
max_seq_length = 2048
Rank=256
sample_len=2000
max_iter_steps=500
###--------------------------------###

train_parameters=f"_lora_fp16_r{Rank}_s{sample_len}_i{max_iter_steps}_msl{max_seq_length}"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/"+base_model,
    #model_name= "./lora/Qwen25_7B_Instruct_lora_fp16_r256_s2000_i2000_msl2048",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    load_in_8bit=load_in_8bit
)

print(model.dtype)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 04-26 16:48:11 [__init__.py:239] Automatically detected platform cuda.
==((====))==  Unsloth 2025.3.18: Fast Llama patching. Transformers: 4.47.1. vLLM: 0.8.2.
   \\   /|    NVIDIA GeForce RTX 4070 Ti SUPER. Num GPUs = 1. Max memory: 15.693 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
torch.bfloat16


In [3]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
      )

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = Rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 123,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.3.18 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [5]:
df_train=pd.read_csv('./drawing-with-llms/svg_score_extend_train1.csv')
df_train.head(2)

,description,response_1,gpt_svg_1,gpt_score_1
0,"'Golden wheat fields under a setting sun',","```svg\n<svg viewBox=""0 0 200 200"" width=""200""...","<svg viewBox=""0 0 200 200"" width=""200"" height=...",0.994973
1,"'Snowy mountains under a clear blue sky',","```svg\n<svg viewBox=""0 0 200 100"" width=""200""...","<svg viewBox=""0 0 200 100"" width=""200"" height=...",0.978844


In [6]:
df_test=pd.read_csv('./drawing-with-llms/svg_score_test_vqa.csv')
df_test.head(2)

,description,gpt_svg_1,gpt_score_sl,response,vqa_pair,response_2,gpt_svg_2
0,"'Vibrant autumn forest',","<svg viewBox=""0 0 200 200"" width=""200"" height=...",0.904117,Here is the visual question answering (VQA) pa...,"{'description': 'Vibrant autumn forest', 'ques...","Here's an improved SVG representation of a ""Vi...","<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."
1,"'Morning dew on grass',","<svg viewBox=""0 0 200 200"" width=""200"" height=...",0.987024,Here is a visual question answering (VQA) pair...,"{'description': 'Morning dew on grass', 'quest...","Here's an improved SVG representation of ""Morn...","<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."


In [7]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}
"""

EOS_TOKEN = tokenizer.eos_token  # Must add EOS_TOKEN

def formatting_prompts_func(examples):
    topics = examples["description"]  # Using 'topic' as instruction
    svgs = examples["gpt_svg_1"]  # Using 'svg_code' as output
    texts = []

    
    for topic, svg_code in zip(topics, svgs):
        # No additional input is needed, so we pass an empty string
        text = alpaca_prompt.format(f"Generate a SVG code for the given input:",topic,svg_code) + EOS_TOKEN
        texts.append(text)

        print(texts)
        
    return { "text": texts }

from datasets import Dataset
import pandas as pd
# Convert DataFrame to Hugging Face Dataset
dataset_train = Dataset.from_pandas(df_train)
dataset_train = dataset_train.map(formatting_prompts_func, batched=True)

dataset_test = Dataset.from_pandas(df_test)
dataset_test = dataset_test.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/1975 [00:00<?, ? examples/s]

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



Map:   0%|          | 0/75 [00:00<?, ? examples/s]

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



['Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nGenerate a SVG code for the given input:\n\n### Input:\n \'Vibrant autumn forest\',\n\n### Response:\n<svg viewBox="0 0 200 200" width="200" height="200" xmlns="http://www.w3.org/2000/svg">\n  <defs>\n    <linearGradient id="autumnGradient" x1="0" y1="0" x2="1" y2="1">\n      <stop offset="0%" stop-color="#FF7F50" />\n      <stop offset="50%" stop-color="#FFD700" />\n      <stop offset="100%" stop-color="#8B4513" />\n    </linearGradient>\n  </defs>\n  <rect x="0" y="0" width="200" height="200" fill="url(#autumnGradient)" />\n  <g transform="translate(50, 150)">\n    <rect x="-10" y="-60" width="20" height="60" fill="#8B4513" />\n    <circle cx="0" cy="-80" r="30" fill="#FF4500" />\n    <circle cx="-25" cy="-100" r="20" fill="#FFD700" />\n    <circle cx="25" cy="-100" r="20" fill="#FF6347" />\n  </g>\n  <g t

In [8]:
# Check dataset sample output
dataset_train['text'][0]

'Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nGenerate a SVG code for the given input:\n\n### Input:\n\'Golden wheat fields under a setting sun\',\n\n### Response:\n<svg viewBox="0 0 200 200" width="200" height="200" xmlns="http://www.w3.org/2000/svg">\n  <!-- Background for the sky -->\n  <rect x="0" y="0" width="200" height="100" fill="orange" opacity="0.7"/>\n  \n  <!-- Sun -->\n  <circle cx="100" cy="50" r="30" fill="yellow" opacity="0.8"/>\n  \n  <!-- Wheat fields -->\n  <rect x="0" y="100" width="200" height="100" fill="goldenrod"/>\n  \n  <!-- Wheat stalks -->\n  <g stroke="saddlebrown" stroke-width="2">\n    <line x1="30" y1="100" x2="30" y2="150"/>\n    <line x1="50" y1="100" x2="50" y2="150"/>\n    <line x1="70" y1="100" x2="70" y2="150"/>\n    <line x1="90" y1="100" x2="90" y2="150"/>\n    <line x1="110" y1="100" x2="110" y2="150"/>\n    <line

In [9]:
import mlflow
mlflow.set_tracking_uri("ml_unsloth_runs")  # Or your remote tracking URI
mlflow.set_experiment("unsloth-lora-experiments")  # Your experiment name

<Experiment: artifact_location='/home/ml_unsloth_runs/990149045631854310', creation_time=1745245436792, experiment_id='990149045631854310', last_update_time=1745245436792, lifecycle_stage='active', name='unsloth-lora-experiments', tags={}>

In [10]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_train,
    eval_dataset = dataset_test,  # Add test dataset here
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 5, 
        max_steps = max_iter_steps,
        learning_rate = 5e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit", # "adamw_torch" better for fp16
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 123,
        eval_strategy = "steps", 
        eval_steps = 10, 
        output_dir = "outputs",
        report_to = "mlflow", # Use this for WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1975 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/75 [00:00<?, ? examples/s]

In [11]:
formatted_model = base_model.replace(".", "").replace("-", "_")
run_name=formatted_model+train_parameters
with mlflow.start_run(run_name=run_name):
    trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,975 | Num Epochs = 3 | Total steps = 500
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 389,021,696/3,601,771,520 (10.80% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
10,0.577800,0.397740
20,0.429500,0.323524
30,0.397200,0.320364
40,0.365600,0.303079
50,0.354600,0.292152
60,0.339300,0.289193
70,0.332300,0.290992
80,0.333000,0.287131
90,0.313100,0.279676
100,0.283400,0.293559


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


In [12]:
# #This ONLY saves the LoRA adapters, and not the full model.
# model.save_pretrained("./lora/lora_model_3b_v3") # Local saving
# tokenizer.save_pretrained("./lora/lora_model_3b_v3")

In [13]:
# import gc
# del model
# gc.collect()
# torch.cuda.empty_cache()

In [14]:
# import psutil

# def kill_large_python_processes():
#     # Loop over all running processes
#     for proc in psutil.process_iter(['pid', 'name', 'memory_info', 'exe']):
#         try:
#             # Check if the process is Python and type is 'C' (for computation)
#             if 'python' in proc.info['name'].lower():
#                 # Check if memory usage is greater than 2048 MB
#                 memory_usage_mb = proc.info['memory_info'].rss / (1024 * 1024)  # Convert bytes to MB
#                 if memory_usage_mb > 2048:
#                     print(f"Killing Python process with PID {proc.info['pid']} using {memory_usage_mb} MB memory")
#                     proc.kill()  # Kill the process
#         except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
#             pass  # Handle processes that might disappear during iteration

# #kill_large_python_processes()

In [15]:
# from unsloth import FastLanguageModel
# model, tokenizer = FastLanguageModel.from_pretrained(
# model_name = "./lora/lora_model_3b_v2", # YOUR MODEL YOU USED FOR TRAINING
# max_seq_length = 2048,
# dtype = (None),
# load_in_4bit = False,
# )
# FastLanguageModel.for_inference(model) # Enable native 2x faster inference


In [16]:
# # alpaca_prompt = You MUST copy from above!
# alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

# ### Instruction:
# {}

# ### Input:
# {}

# ### Response:
# {}"""

# inputs = tokenizer(
# [
#     alpaca_prompt.format(
#         "Please write a SVG code fo rthe given topic?", # instruction
#         "Golden sun rising in the east", # input
#         "", # output - leave this blank for generation!
#     )
# ], return_tensors = "pt").to("cuda")

# outputs = model.generate(**inputs, max_new_tokens = 1024, use_cache = True)
# tokenizer.batch_decode(outputs)

In [17]:
#save merged 16bit
import os
dir_path = "./lora/"+run_name
os.makedirs(dir_path, exist_ok=True)
model.save_pretrained_merged(dir_path, tokenizer, save_method = "merged_16bit")

Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 12.16 out of 31.21 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


 57%|████████████████████████████████████████████████████████████████████████████████████                                                               | 16/28 [00:00<00:00, 77.43it/s]
We will save to Disk and not RAM now.
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 28/28 [00:01<00:00, 14.27it/s]


Unsloth: Saving tokenizer... Done.
Done.
